# 面向对象高级与学生管理系统

本篇对应《Python进阶（大模型语言进阶）》第 2 天，包含两部分：

1. 面向对象高级语法：类的定义方式、继承、封装、多态、抽象类，以及对象属性、类属性、类方法和静态方法。
2. 面向对象综合案例：把学生信息管理系统拆成 `Student`、`StudentCMS` 和 `main.py`，实现学生信息的增删改查、保存、加载和退出。

课程代码以 Markdown 代码围栏保留，重点代码与配套源码保持一致。

## 一、类的三种定义方式

课程回顾了三种写法：

```python
class Teacher:
    pass

class Teacher():
    pass

class Teacher(object):
    pass
```

Python 中所有类默认继承顶级基类 `object`。课程后续统一推荐显式写出 `class 类名(object):`，这样能直接看出类的基类，也与继承语法保持一致。

## 二、继承

### 1. 概念和作用

继承指子类从已有父类获得属性和方法。站在已有类的角度也叫派生：从一个已有类产生新类。父类也叫基类，子类也叫派生类或扩展类。继承关系描述的是类之间的关系，不是对象之间的关系。

现实中的层级可以抽象为“生物 → 动物 → 哺乳动物 → 人类”等。代码中的继承主要解决共有功能重复书写的问题，提高代码复用率。所有 Python 类最终都继承自 `object`。

基本语法：

```python
class 父类名(object):
    ...

class 子类名(父类名):
    ...
```

### 2. 单继承

单继承表示一个子类只继承一个父类。子类不用重新编写父类已有的属性和方法：

```python
class Father(object):
    def __init__(self):
        self.sex = "男"

    def walk(self):
        print("喜欢散步行走...")

class Son(Father):
    pass

son = Son()
print(son.sex)
son.walk()
```

“摊煎饼”示例中，`Prentice(Master)` 可以直接复用师父的配方和制作方法。

### 3. 多继承与 MRO

多继承表示一个类同时继承多个父类：

```python
class Father(object):
    pass

class Mother(object):
    pass

class Son(Father, Mother):
    pass
```

如果多个父类有同名属性或方法，Python 会按方法解析顺序（MRO）查找，优先使用继承列表中靠前的父类：

```python
class Master(object):
    def __init__(self):
        self.kongfu = "[古法煎饼果子配方]"

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

class School(object):
    def __init__(self):
        self.kongfu = "[黑马AI煎饼果子配方]"

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

class Prentice(School, Master):
    pass

xiaoming = Prentice()
print(xiaoming.kongfu)
xiaoming.make_cake()

print(Prentice.__mro__)
print(Prentice.mro())
```

`__mro__` 属性和 `mro()` 方法都可以查看查找顺序。

### 4. 重写父类同名属性和方法

子类定义与父类同名的属性或方法，就会覆盖继承来的成员，调用时遵循“就近原则”：先找子类，再按 MRO 继续找父类。

```python
class Prentice(School, Master):
    def __init__(self):
        self.kongfu = "[独创煎饼果子配方]"

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

p = Prentice()
print(p.kongfu)
p.make_cake()
```

这段代码体现了子类自己的配方和制作方法优先于两个父类。

### 5. 子类重写后调用父类功能

子类想保留自己的实现，同时调用指定父类，可以直接使用“父类名 + 方法名 + `self`”：

```python
class Prentice(School, Master):
    def __init__(self):
        self.kongfu = "[独创煎饼果子配方]"

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

    def make_master_cake(self):
        Master.__init__(self)
        Master.make_cake(self)

    def make_school_cake(self):
        School.__init__(self)
        School.make_cake(self)

p = Prentice()
print(p.kongfu)
p.make_cake()
p.make_master_cake()
p.make_school_cake()
print("-" * 34)
p.make_cake()
```

父类的初始化方法会重新给当前对象的 `kongfu` 赋值，因此最后一次 `make_cake()` 看到的是最近一次父类初始化后的配方。

### 6. 使用 `super()` 和多层继承

在单继承中，可以用 `super()` 自动找到父类：

```python
class Prentice(School):
    def __init__(self):
        self.kongfu = "[独创煎饼果子配方]"

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

    def make_old_cake(self):
        super().__init__()
        super().make_cake()
```

课程提示：`super()` 适合单继承；多继承时如果要精确选择某个父类，直接使用父类名更直观。

多层继承是“子类继续被另一个类继承”，例如 `TuSun(Prentice)`，它可以获得 `Prentice`、`Master` 和 `School` 沿链路提供的功能：

```python
class TuSun(Prentice):
    pass

ts = TuSun()
ts.make_cake()
ts.make_master_cake()
ts.make_school_cake()
```

## 三、封装与私有成员

### 1. 封装概念

封装就是把属性和方法写在类中，并通过访问权限隐藏实现细节，只对外提供公共访问方式。课程强调它的两个作用：用私有化提高代码安全性，用函数封装提高代码复用性。代价是需要额外编写公共接口，代码量会增加。

在属性名或方法名前加两个下划线即可设置私有成员：

```python
__属性名
def __方法名(self):
    ...
```

私有成员只能在类内部使用，外部或子类不能直接访问。需要在外部使用时，应通过公共方法访问。

### 2. 私有属性的获取和修改

```python
class Prentice:
    def __init__(self):
        self.kongfu = "[黑马煎饼果子配方]"
        self.__money = 20000

    def make_cake(self):
        print(f"运用{self.kongfu}制作煎饼果子")

    def get_money(self):
        return self.__money

    def set_money(self, money):
        self.__money = money

class TuSun(Prentice):
    pass

ts = TuSun()
print(ts.kongfu)
ts.make_cake()
# print(ts.__money)  # 私有属性不能直接访问
ts.set_money(100)
print(ts.get_money())
```

课程约定使用 `get_xx` 获取私有属性、`set_xx` 修改私有属性。实际项目中，公共接口还可以在赋值时增加校验。

### 3. 私有方法

私有方法的形式是 `def __方法名(self):`。外部不能直接调用它，可以在类内部再提供一个公共方法作为入口：

```python
class Prentice:
    def __make_cake(self):
        print("制作独创配方煎饼果子")

    def make(self):
        self.__make_cake()

p = Prentice()
p.make()
# p.__make_cake()  # 不能从类外部直接访问
```

## 四、多态与抽象类

### 1. 多态的含义和成立条件

多态指同一个函数接收不同对象时，表现出不同效果。要实现课程所讲的多态，需要：

1. 有继承关系；
2. 子类重写父类方法；
3. 把子类对象传给面向父类设计的调用函数。

```python
class Animal:
    def speak(self):
        pass

class Dog(Animal):
    def speak(self):
        print("狗叫: 汪汪汪")

class Cat(Animal):
    def speak(self):
        print("猫叫: 喵喵喵")

def make_noise(an: Animal):
    an.speak()

dog = Dog()
cat = Cat()
make_noise(dog)
make_noise(cat)
```

调用者只依赖 `Animal` 约定的 `speak()`，实际传入 `Dog` 或 `Cat` 后，由对应子类决定具体行为。课程还演示了“汽车类”只要也有 `speak()` 方法，就可以被同一平台调用，这体现了 Python 的鸭子类型特点。

### 2. 多态战斗平台案例

需求是：一代英雄战斗力 `60`、二代英雄战斗力 `80`，敌机战斗力 `70`。先写一个公共的对战平台，再在不修改平台代码的情况下传入不同英雄：

```python
class HeroFighter:
    def power(self):
        return 60

class AdvHeroFighter(HeroFighter):
    def power(self):
        return 80

class EnemyFighter:
    def power(self):
        return 70

# 公共平台：接收不同对象，调用同名方法
def object_play(hero, enemy):
    if hero.power() >= enemy.power():
        print("英雄机 战胜 敌机!")
    else:
        print("英雄机 惜败 敌机!")

h1 = HeroFighter()
h2 = AdvHeroFighter()
e1 = EnemyFighter()
object_play(h1, e1)
object_play(h2, e1)
```

如果不使用多态，每增加一个英雄型号就要重复编写比较和输出逻辑。使用多态后，平台函数相当于调用者，子类相当于任务实现者，双方可以解耦；后来新增子类只要遵守父类方法约定，平台不用修改。

### 3. 抽象类（接口）

抽象类包含抽象方法。课程示例用 `pass` 表示没有具体实现的方法，父类用来制定标准，子类负责完成具体实现：

```python
class AC:
    def cool_wind(self):
        pass

    def hot_wind(self):
        pass

    def swing_l_r(self):
        pass

class XiaoMi(AC):
    def cool_wind(self):
        print("小米核心制冷技术!")

    def hot_wind(self):
        print("小米核心制热技术!")

    def swing_l_r(self):
        print("小米空调静音左右摆风技术!")

class Gree(AC):
    def cool_wind(self):
        print("格力核心制冷技术!")

    def hot_wind(self):
        print("格力核心制热技术!")

    def swing_l_r(self):
        print("格力空调低频左右摆风技术!")
```

抽象类适合做顶层设计和行业规范，要求子类实现约定的方法，再配合多态得到不同厂商的工作状态。本课使用的是概念性抽象类写法，未引入 `abc` 模块。

## 五、对象属性、类属性、类方法和静态方法

### 1. 对象属性与类属性

对象属性属于某一个对象，通常写在 `__init__()` 中，每个对象可以有不同值；类属性写在类中、方法外，由该类的所有对象共享。

```python
class Student:
    teacher_name = "水镜先生"  # 类属性

    def __init__(self, name, age):
        self.name = name        # 对象属性
        self.age = age

    def __str__(self):
        return "姓名: %s, 年龄: %d" % (self.name, self.age)

s1 = Student("曹操", 38)
s2 = Student("曹操", 38)
s1.name = "许褚"
s1.age = 40
print(s1)
print(s2)

print(Student.teacher_name)
print(s1.teacher_name)
Student.teacher_name = "夯哥"
print(s1.teacher_name)
print(s2.teacher_name)
```

通过 `对象名.类属性` 可以读取类属性，但使用 `对象名.类属性 = 值` 时，通常会给该对象新建同名对象属性，并不会修改类属性。修改共享的类属性应使用 `类名.类属性 = 值`。

### 2. 类方法和静态方法

类方法属于类，使用 `@classmethod` 标识，第一个参数必须接收类对象，约定命名为 `cls`。静态方法使用 `@staticmethod` 标识，不要求接收 `self` 或 `cls`。两者都可以通过类名或对象名调用，课程推荐使用类名调用。

```python
class Student:
    school = "黑马程序员"

    @classmethod
    def show1(cls):
        print(f"cls: {cls}")
        print(cls.school)
        print("我是类方法")

    @staticmethod
    def show2():
        print(Student.school)
        print("我是静态方法")

s1 = Student()
s1.show1()
s1.show2()
```

判断方式：方法需要使用类对象或访问类属性时使用类方法；既不需要对象也不需要类对象时使用静态方法。

### 3. `__dict__` 扩展属性

`__dict__` 是 Python 内置属性，可以把实例属性转换为字典：

```python
class A(object):
    a = 0

    def __init__(self):
        self.b = 1

aa = A()
print(A.__dict__)
print(aa.__dict__)
```

`类名.__dict__` 包含类的属性和方法，`对象.__dict__` 主要包含该实例的属性和值。在学生管理系统中，它用于把 `[Student 对象, ...]` 转成 `[字典, ...]`，便于写入文本文件，也可以用字典解包重新创建对象：

```python
my_dict = {"name": "德桦", "gender": "男", "age": 81, "phone": "111", "desc": "刻骨铭心"}
student = Student(**my_dict)
```

## 六、综合案例：学生信息管理系统

### 1. 需求和角色分析

系统使用面向对象、字符串、列表、字典和文件知识，完成学生信息的：

- 添加、删除、修改；
- 查询单个学生、显示所有学生；
- 保存和加载信息；
- 退出系统。

系统中有两个角色：`Student` 学生类和 `StudentCMS` 学生管理系统类。为了便于维护，一个角色使用一个程序文件，并用 `main.py` 作为程序入口。

项目文件规划：

```text
StudentManagerSystem/
├── main.py
├── student.py
├── studentcms.py
└── stu_data.txt
```

### 2. `student.py`：封装学生对象

学生属性为姓名、性别、年龄、手机号和描述信息。`__init__()` 初始化属性，`__str__()` 负责把对象转换为可读文本：

```python
class Student:
    def __init__(self, name, gender, age, phone, desc):
        self.name = name
        self.gender = gender
        self.age = age
        self.phone = phone
        self.desc = desc

    def __str__(self):
        return (f"姓名: {self.name}, 性别: {self.gender}, "
                f"年龄: {self.age}, 手机号: {self.phone}, "
                f"描述信息: {self.desc}")

if __name__ == "__main__":
    s = Student("乔峰", "男", 38, "13112345678", "丐帮帮主")
    print(s)
```

### 3. `StudentCMS` 的框架和菜单

管理系统在 `__init__()` 中维护一个学生对象列表：

```python
from student import Student
import time

class StudentCMS(object):
    def __init__(self):
        self.stu_list = []

    @staticmethod
    def show_view():
        print("*" * 23)
        print("学生管理系统V2.0版")
        print("\t1.添加学生信息")
        print("\t2.删除学生信息")
        print("\t3.修改学生信息")
        print("\t4.查询单个学生信息")
        print("\t5.查询所有学生信息")
        print("\t6.保存学生信息")
        print("\t0.退出系统")
        print("*" * 23)
```

菜单函数没有使用 `self`，所以源码把它定义为静态方法。系统入口 `start()` 通过死循环重复显示菜单，接收字符串编号，再用 `if/elif` 分派到各个业务函数。输入 `0` 后还要用 `Y/N` 二次确认，只有输入 `y` 或 `Y` 才退出。

### 4. 添加、删除、修改和查询

**添加**：收集用户输入，创建 `Student` 对象，再追加到 `stu_list`。

```python
def add_student(self):
    name = input("请输入学生姓名:")
    gender = input("请输入学生性别:")
    age = int(input("请输入学生年龄:"))
    phone = input("请输入学生电话:")
    desc = input("请输入学生描述信息:")
    stu = Student(name, gender, age, phone, desc)
    self.stu_list.append(stu)
    print(f"添加 {name} 学生信息成功!\n")
```

**删除**：按姓名遍历列表，找到后移除；`for...else` 中的 `else` 只在没有执行 `break` 时运行。

```python
def del_student(self):
    del_name = input("请输入要删除的学生姓名:")
    for stu in self.stu_list:
        if stu.name == del_name:
            self.stu_list.remove(stu)
            print(f"学员 {del_name} 信息删除成功!\n")
            break
    else:
        print("查无此人, 请检查后重新删除!\n")
```

**修改**：先按姓名找到对象，再修改其余属性。**查询单个**同样按姓名找到后打印对象。**查询所有**先判断列表是否为空，再遍历打印。

### 5. 保存和加载学生信息

保存时不能把对象的内存地址直接写进文本文件，要先用 `__dict__` 转成字典列表，再转成字符串：

```python
def save_student(self):
    with open("./stu_data.txt", "w", encoding="utf-8") as dest_f:
        stu_dict = [stu.__dict__ for stu in self.stu_list]
        dest_f.write(str(stu_dict))
```

加载时读取字符串，用 `eval()` 转成列表，再用字典解包还原 `Student` 对象：

```python
def load_student(self):
    try:
        with open("./stu_data.txt", "r", encoding="utf-8") as src_f:
            stu_data = src_f.read()
            stu_list = eval(stu_data)
            if len(stu_list) == 0:
                stu_list = []
            self.stu_list = [Student(**stu_dict) for stu_dict in stu_list]
    except:
        with open("./stu_data.txt", "w", encoding="utf-8"):
            pass
```

首次运行时文件可能不存在，源码用 `try/except` 创建空文件。若文件存在但为空，必须先把空内容处理成空列表，否则直接 `eval("")` 会报错。课程源码使用 `eval()` 读取本地自己写入的字典列表，真实项目处理不可信文件时应采用安全的数据格式解析方式。

### 6. 程序入口和完整运行流程

`main.py` 只负责创建系统对象并启动：

```python
from studentcms import StudentCMS

if __name__ == "__main__":
    stu_cms = StudentCMS()
    stu_cms.start()
```

`start()` 的关键顺序是：

1. 调用 `load_student()`，恢复上次保存的信息；
2. 循环显示菜单；
3. 读取操作编号并调用对应方法；
4. 保存时写入 `stu_data.txt`；
5. 输入 `0` 后再次确认，确认退出前自动保存一次。

源码还用 `time.sleep(1)` 让菜单显示更明显。学生管理系统框架文件中的业务方法最初是 `pass` 占位，完整源码则依次补齐了这些操作。

## 七、本篇易错点和要点回顾

- 多继承同名成员按 MRO 查找，继承列表中靠前的父类优先。
- 子类重写父类成员后，调用优先使用子类实现；想精确调用某个父类时使用 `父类名.方法(self)`。
- `super()` 会沿继承关系自动查找，课程建议主要用于单继承。
- 私有成员名前加 `__`，子类和类外不能直接访问，要通过公共接口。
- 多态依赖继承、方法重写和父类引用指向子类对象，公共平台因此可以复用并扩展。
- 对象属性属于实例，类属性由整个类共享；修改类属性使用 `类名.属性`。
- 类方法使用 `@classmethod` 和 `cls`，静态方法使用 `@staticmethod`，不要求特殊参数。
- 学生对象持久化要经过“对象 → `__dict__` → 字典列表字符串”，加载则反向恢复。
- 文件首次不存在或内容为空时，要先处理异常和空列表，再进行类型转换。